In  a docker terminal run
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

then open jupyter in the printed link (such as http://127.0.0.1:8888/tree?token=4d9d578397aede50fc5bd2d92794b562a6bbcbc73b2dfe51)

CODE HERE, REFRESH AND EXECUTE IN THE BROWSER

In [1]:
import os
import sys
import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

sys.path.append('/app')

os.environ['DJANGO_SETTINGS_MODULE'] = 'app.settings'
os.environ['PYTHONPATH'] = '/app'

django.setup()

In [2]:

from toolbox import models, views
import shapefile  # pyshp
import tempfile
import zipfile
import os
from django.http import HttpResponse
from shapely.geometry import shape
from shapely.ops import transform
import json



In [28]:
lakes = models.Lake.objects.filter(pk__lte=20)

In [4]:
fc = views.create_feature_collection(lakes)


In [5]:
permdir = os.mkdir('permdir')

FileExistsError: [Errno 17] File exists: 'permdir'

In [51]:
permdir

In [45]:
tmpdir = tempfile.mkdtemp()

In [46]:
tmpdir

'/tmp/tmp5b6f04td'

In [5]:
shp_path = os.path.join('permdir', "lakes_v5")

In [6]:
writer = shapefile.Writer(shp_path)
#writer.autoBalance = 1

In [7]:
writer_fields = models.Lake.shp_writer_fields()
writer_fields

{'id': {'field_type': 'N', 'decimal': 0},
 'name': {'field_type': 'C', 'decimal': 0},
 'fgw_id': {'field_type': 'N', 'decimal': 4},
 'shape_area': {'field_type': 'F', 'decimal': 4},
 'minimum_environmental_flow': {'field_type': 'F', 'decimal': 2},
 'min_surplus_volume': {'field_type': 'F', 'decimal': 4},
 'mean_surplus_volume': {'field_type': 'F', 'decimal': 0},
 'max_surplus_volume': {'field_type': 'F', 'decimal': 0},
 'plus_days': {'field_type': 'F', 'decimal': 0}}

In [8]:
for key, val in writer_fields.items():
    writer.field(key[:10], val['field_type'], decimal=val['decimal'])


In [9]:
for feature in fc["features"]:
    geom = shape(feature["geometry"])
    props = feature["properties"]

    writer.shape(geom.__geo_interface__)

    # collect all values in the same order as writer_fields
    record_values = [props.get(field) or 0 if writer_fields[field]['field_type'] in ('N', 'F') else props.get(field, '') 
                     for field in writer_fields]
    
    writer.record(*record_values)



writer.close()

In [10]:

with open(f"{shp_path}.prj", "w") as f:
    f.write("""GEOGCS["WGS 84",
        DATUM["WGS_1984",
        SPHEROID["WGS 84",6378137,298.257223563]],
        PRIMEM["Greenwich",0],
        UNIT["degree",0.0174532925199433]]""")



In [11]:
    # --- 6. Zip the shapefile -----------------------------------------
    zip_path = os.path.join(tmpdir, "lakes_v2.zip")
    with zipfile.ZipFile(zip_path, 'w') as z:
        for ext in ["shp", "shx", "dbf", "prj"]:
            z.write(f"{shp_path}.{ext}", arcname=f"sinks_v1.{ext}")

    # --- 7. Return as HTTP response ------------------------------------
    with open(zip_path, "rb") as fh:
        response = HttpResponse(fh.read(), content_type="application/zip")
        response["Content-Disposition"] = "attachment; filename=sinks.zip"
        return response


NameError: name 'tmpdir' is not defined

In [52]:
A = 'ASDFGHJKLÖÄ'
A[:10]

'ASDFGHJKLÖ'

In [18]:
lakes.model.shp_writer_fields()

{'id': {'field_type': 'N', 'decimal': 0},
 'name': {'field_type': 'C', 'decimal': 0},
 'fgw_id': {'field_type': 'N', 'decimal': 4},
 'shape_area': {'field_type': 'F', 'decimal': 4},
 'minimum_environmental_flow': {'field_type': 'F', 'decimal': 2},
 'min_surplus_volume': {'field_type': 'F', 'decimal': 4},
 'mean_surplus_volume': {'field_type': 'F', 'decimal': 0},
 'max_surplus_volume': {'field_type': 'F', 'decimal': 0},
 'plus_days': {'field_type': 'F', 'decimal': 0}}

In [31]:
str(lakes.model.__name_de__())

AttributeError: type object 'Lake' has no attribute '__name_de__'

In [3]:
sink = models.SiekerSink.objects.get(id=300)

In [5]:
sink.to_point_feature(epsg=25833)

{'type': 'Feature',
 'geometry': {'type': 'Point',
  'coordinates': [404718.36006031785, 5811657.867294821]},
 'properties': {'id': 300,
  'name': 'Senke',
  'depth': 1.14,
  'area': 776,
  'volume': 358,
  'avg_depth': 0.46,
  'max_elevation': 34.6,
  'min_elevation': 33.5,
  'urbanarea_percent': 100.0,
  'wetlands_percent': 0.0,
  'distance_t': 263,
  'dist_lake': '> 500 m',
  'waterdist': '> 500 m',
  'umsetzbark': 'schwierig',
  'index_feasibility': 0,
  'distance_lake': 9498,
  'nearest_lake': None,
  'distance_stream': 365,
  'nearest_stream': None}}